In [1]:
#EKSPORT DATA RINGKAS DARI PYTHON
import os
import sqlite3
import numpy as np
import pandas as pd
from statsmodels.stats.proportion import proportions_ztest, proportion_confint

os.makedirs("../dashboard/exports", exist_ok=True)
con = sqlite3.connect("../data/web_funnel_events.sqlite")

# 1) Funnel
f = pd.read_sql("""
SELECT
  COUNT(DISTINCT session_id) AS s0,
  COUNT(DISTINCT CASE WHEN event_name='view_product'   THEN session_id END) AS s1,
  COUNT(DISTINCT CASE WHEN event_name='add_to_cart'    THEN session_id END) AS s2,
  COUNT(DISTINCT CASE WHEN event_name='begin_checkout' THEN session_id END) AS s3,
  COUNT(DISTINCT CASE WHEN event_name='purchase'       THEN session_id END) AS s4
FROM events""", con).iloc[0].values
funnel = pd.DataFrame({
    "stage_order": [1, 2, 3, 4, 5],
    "stage": ["Sessions Started", "Viewed Product", "Added to Cart",
              "Began Checkout", "Purchased"],
    "sessions": f})
funnel["step_conversion_pct"] = (funnel["sessions"] / funnel["sessions"].shift(1) * 100).round(2)
funnel["drop_off_pct"] = (100 - funnel["step_conversion_pct"]).round(2)
funnel["overall_conversion_pct"] = (funnel["sessions"] / funnel["sessions"].iloc[0] * 100).round(2)
funnel.to_csv("../dashboard/exports/funnel.csv", index=False)

# 2) Segments
sessions = pd.read_sql("SELECT * FROM sessions", con, parse_dates=["started_at"])
flags = pd.read_sql("""
SELECT session_id,
  MAX(event_name='view_product') AS viewed, MAX(event_name='add_to_cart') AS carted,
  MAX(event_name='begin_checkout') AS checkout, MAX(event_name='purchase') AS purchased
FROM events GROUP BY session_id""", con)
df = sessions.merge(flags, on="session_id", how="left")
cols = ["viewed", "carted", "checkout", "purchased"]
df[cols] = df[cols].fillna(0).astype(int)
df["month"] = df["started_at"].dt.to_period("M").astype(str)

parts = []
for col, label in [("device", "Device"), ("referrer", "Traffic Source"), ("month", "Month")]:
    g = df.groupby(col).agg(sessions=("session_id", "count"),
                            purchased=("purchased", "sum")).reset_index()
    g = g.rename(columns={col: "segment"})
    g.insert(0, "segment_type", label)
    g["conversion_pct"] = (g["purchased"] / g["sessions"] * 100).round(2)
    parts.append(g)
segments = pd.concat(parts, ignore_index=True)
segments["overall_avg_pct"] = round(df["purchased"].mean() * 100, 2)
segments.to_csv("../dashboard/exports/segments.csv", index=False)

# 3) A/B test
ab = pd.read_csv("../data/marketing_AB.csv").drop(columns=["Unnamed: 0"])
ab.columns = [c.replace(" ", "_") for c in ab.columns]
ab["converted"] = ab["converted"].astype(int)

s = ab.groupby("test_group")["converted"].agg(users="count", conversions="sum").reset_index()
s["group_label"] = s["test_group"].map({"ad": "Ad (Treatment)", "psa": "PSA (Control)"})
s["conversion_pct"] = (s["conversions"] / s["users"] * 100).round(3)
ci = [proportion_confint(c, n, method="wilson") for c, n in zip(s["conversions"], s["users"])]
s["ci_low_pct"] = [round(lo * 100, 3) for lo, hi in ci]
s["ci_high_pct"] = [round(hi * 100, 3) for lo, hi in ci]
s.to_csv("../dashboard/exports/ab_summary.csv", index=False)

order = {"Monday": 1, "Tuesday": 2, "Wednesday": 3, "Thursday": 4,
         "Friday": 5, "Saturday": 6, "Sunday": 7}
rows = []
for day, g in ab.groupby("most_ads_day"):
    a = g.loc[g["test_group"] == "ad", "converted"]
    b = g.loc[g["test_group"] == "psa", "converted"]
    _, p = proportions_ztest([a.sum(), b.sum()], [len(a), len(b)])
    rows.append({"day": day, "day_order": order[day],
                 "ad_conversion_pct": round(a.mean() * 100, 3),
                 "psa_conversion_pct": round(b.mean() * 100, 3),
                 "lift_pp": round((a.mean() - b.mean()) * 100, 3), "p_value": p})
by_day = pd.DataFrame(rows).sort_values("day_order")
by_day["p_bonferroni"] = (by_day["p_value"] * len(by_day)).clip(upper=1)
by_day["significant"] = np.where(by_day["p_bonferroni"] < 0.05,
                                 "Significant", "Not significant")
by_day.round(6).to_csv("../dashboard/exports/ab_by_day.csv", index=False)

print(sorted(os.listdir("../dashboard/exports")))

['ab_by_day.csv', 'ab_summary.csv', 'funnel.csv', 'segments.csv']
